# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR^2 Croissant dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, columns, and their `@id`s.

Below, we list all record sets contained in the schema, along with their field names and `@id`s.

In [ ]:
# List all available record sets and their fields by @id
record_sets = list(dataset.record_sets)
print(f"Found {len(record_sets)} record set(s) in this dataset.")

recordset_ids = []
for rs in record_sets:
    print(f"RecordSet name: {rs.name}, @id: {rs.id}")
    recordset_ids.append(rs.id)
    print("  Fields:")
    for f in rs.fields:
        print(f"    Field name: {f.name}, @id: {f.id}, data type: {f.data_type}")
    print("--")

## 3. Data Extraction
Load data from specified record set(s) into a DataFrame for analysis. We reference sets and fields by their `@id`.

In [ ]:
# For demonstration, we extract all record sets
dataframes = {}

for record_set_id in recordset_ids:
    # Load records for each record set
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} records for RecordSet @id={record_set_id}")
    print(f"Columns: {df.columns.tolist()}")
    print(df.head(2))
    print("---\n")

# Example: Show columns for the first record set
if recordset_ids:
    main_record_set = recordset_ids[0]
    print(f"RecordSet for EDA: {main_record_set}")
    print(f"Columns: {dataframes[main_record_set].columns.tolist()}")
    display(dataframes[main_record_set].head())

## 4. Exploratory Data Analysis (EDA)
Apply typical processing: filtering, normalizing, categorizing, grouping, and removing outliers.

Here we:
- Choose a numeric field for simple statistics
- Filter records with value above a threshold
- Normalize selected numeric field
- Group by a secondary field if available

In [ ]:
# Select first numeric field for EDA based on its dtype
import numpy as np

df = dataframes[main_record_set]

numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
print(f"Numeric columns detected: {numeric_cols}")

if numeric_cols:
    numeric_field = numeric_cols[0]  # Use the first numeric column

    threshold = df[numeric_field].mean() if df[numeric_field].notnull().any() else 10
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered {len(filtered_df)} records with {numeric_field} > {threshold:.3f}:")
    display(filtered_df.head())

    norm_col = f"{numeric_field}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, norm_col]].head())

    # Optionally group by second column if available
    group_field = None
    for col in df.columns:
        if col != numeric_field and df[col].dtype == object:
            group_field = col
            break

    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
        print(f"Grouped filtered data by {group_field} (mean {numeric_field}):")
        display(grouped_df.head())
else:
    print("No numeric columns found for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields. The plot type depends on the field detected in EDA.

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

if numeric_cols:
    plt.figure(figsize=(6, 4))
    df[numeric_field].hist(bins=30)
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.title(f'Distribution of {numeric_field}')
    plt.show()

    if group_field and group_field in df.columns:
        plt.figure(figsize=(8,4))
        df.boxplot(column=numeric_field, by=group_field)
        plt.title(f'{numeric_field} by {group_field}')
        plt.suptitle('')
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.show()

## 6. Conclusion
This notebook demonstrated how to explore a protein abundance dataset using `mlcroissant`:

- Loaded FAIR^2 dataset metadata and extracted record sets
- Reviewed record set structure and identified usable fields
- Performed filtering, normalization, grouping, and simple visualization on numeric data

For further analysis, consider integrating domain-specific visualizations or machine learning on the DataFrames.

**References**:
- Dataset: [https://sen.science/doi/10.71728/senscience.vq4a-28xa](https://sen.science/doi/10.71728/senscience.vq4a-28xa)
- `mlcroissant` documentation: [https://mlcommons.github.io/croissant/python/](https://mlcommons.github.io/croissant/python/)